# Twitter Sentiment Analysis

In [ ]:
#!pip install gensim --upgrade
#!pip install keras --upgrade
#!pip install pandas --upgrade

In [ ]:
# DataFrame
import pandas as pd

# Matplot
import matplotlib.pyplot as plt
%matplotlib inline

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Utility
import re
import numpy as np
import os
from collections import Counter
import logging
import time
import pickle
import itertools

# Set log
# logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

# Read Dataset

## Dataset details
* **target**: the polarity of the tweet (0 = negative, 2 = neutral, 4 = positive)
* **ids**: The id of the tweet ( 2087)
* **date**: the date of the tweet (Sat May 16 23:58:44 UTC 2009)
* **flag**: The query (lyx). If there is no query, then this value is NO_QUERY.
* **user**: the user that tweeted (robotickilldozr)
* **text**: the text of the tweet (Lyx is cool)

In [ ]:
# # DATASET
DATASET_COLUMNS = ["target", "ids", "date", "flag", "user", "text"]
DATASET_ENCODING = "ISO-8859-1"
TRAIN_SIZE = 0.8

dataset_filename = "training.1600000.processed.noemoticon.csv"
print("Open file:", dataset_filename)
df = pd.read_csv(dataset_filename, encoding =DATASET_ENCODING , names=DATASET_COLUMNS)

In [ ]:
print("Dataset size:", len(df))

In [ ]:
df.head(5)

In [ ]:
df.target.value_counts()

## Map target label to String
* **0** -> **NEGATIVE**
* **2** -> **NEUTRAL**
* **4** -> **POSITIVE**

In [ ]:
decode_map = {0: "NEGATIVE", 2: "NEUTRAL", 4: "POSITIVE"}
def decode_sentiment(label):
    return decode_map[int(label)]

In [ ]:
%%time
df.target = df.target.apply(lambda x: decode_sentiment(x))

In [ ]:
target_cnt = Counter(df.target)

plt.figure(figsize=(16,8))
plt.bar(target_cnt.keys(), target_cnt.values())
plt.title("Dataset labels distribuition")
target_cnt

# Pre-Process dataset

## Stop words and stemming

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import joblib

nltk.download('stopwords')
STOP_WORDS = stopwords.words("english")
STEMMER = SnowballStemmer("english")
print(len(STOP_WORDS))

In [ ]:
# Dump stop_words for usage in pipeline
# joblib.dump(STOP_WORDS, 'stop_words.pkl')
sorted(STOP_WORDS)

In [ ]:
# TEXT CLENAING
TEXT_CLEAN_RE = "@\S+|https?:\S+|http?:\S|[^A-Za-z0-9]+"

def preprocess(text, stem=False):
    # Remove link,user and special characters
    text = re.sub(TEXT_CLEAN_RE, ' ', str(text).lower()).strip()
    tokens = []
    for token in text.split():
        # REMOVE STOP WORDS
        if token not in STOP_WORDS:
            if stem:
                # APPLY STEMMING
                tokens.append(STEMMER.stem(token))
            else:
                tokens.append(token)
    # REJOIN ALL TOKENS
    return " ".join(tokens)

def preprocess_all(text_list):
    # Convert text_list to list (if single string instance)
    if not isinstance(text_list, list):
        text_list = [str(text_list)]
    # Return list of processed UNTOKENIZED strings
    return [preprocess(xi) for xi in text_list]

preprocess("Carl is the greatest Data Scientist of all time")

In [ ]:
# Pre transform
df.text[0:10]

In [ ]:
%%time
df.text = df.text.apply(lambda x: preprocess(x))

In [ ]:
# Post transform
df.text[0:10]

## Split train and test

In [ ]:
df_train, df_test = train_test_split(df, test_size=1-TRAIN_SIZE, random_state=42)
print("TRAIN size:", len(df_train))
print("TEST size:", len(df_test))

## Vectorize

Lets start with CountVectorization:

In [ ]:
sample_txt = df_train.text.iloc[20:25].values
sample_txt

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
sample_matrix = cv.fit_transform(sample_txt)
sample_matrix.A

In [ ]:
sample_matrix.A[0]

In [ ]:
cv.vocabulary_

The problem with simple count vectorization is that word frequency is often not that useful, as words like 'this', 'a'  occur very frequently across all documents. 

**Hence, we also want a measure of how unique a word is i.e. how infrequently the word occurs across all documents (inverse document frequency or idf).** So TFIDF is the product of two components:
1. **TF: term frequency** - how many times the word appears in the document
2. **IDF: Inverse document frequency:** How many times the word appears in any document

**EXAMPLE:** 

Consider a document containing **100 words wherein the word sun appears 3 times.** The term frequency (i.e., tf) for sun is then (3 / 100) = 0.03. Now, assume we have **10 million documents and the word sun appears in one thousand of these.** Then, the inverse document frequency (i.e., idf) is calculated as log(10,000,000 / 1,000) = 4 (with log base = 4). Thus, the Tf-idf weight is the product of these quantities: 0.03 * 4 = 0.12.

- TF = 3/100 = 0.03
- IDF = log(10,000,000/1,000) = 9.2
- TF IDF = 0.03 * 4 = 0.276

In essence, less frequent words get weighted higher. 

In [ ]:
%%time
from sklearn.feature_extraction.text import TfidfVectorizer

max_feats = 500000
tfidf_vectorizer = TfidfVectorizer(decode_error='ignore', ngram_range=(1,2), max_features=max_feats)

# Fit_transofrm tfidf
X_train = tfidf_vectorizer.fit_transform(df_train['text'])
X_test = tfidf_vectorizer.transform(df_test['text'])

print("X TRAIN:", X_train.shape)
print("X TEST:", X_test.shape)
print("VOCAB LEN:", len(tfidf_vectorizer.vocabulary_))

In [ ]:
X_train

## PCA

In [ ]:
%%time
# Try PCA?

from sklearn.decomposition import PCA

decomp = PCA()
decomp.fit_transform(X_train[0:110].A)

In [ ]:
# Assuming linear time complexity
rt = 9.77
sample_n = 110
m = X_train.shape[0]
rt_hrs = (m/sample_n)*rt/60/60
print("expected runtime: {:.3f} hrs".format(rt_hrs))

## Label Encoder 

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_train = encoder.fit_transform(df_train.target)
y_test = encoder.fit_transform(df_test.target)

print("CLASSES:", encoder.classes_)
print("Y SHAPE:",  y_train.shape)
y_train[0:5]

# Train & Evaluate

## Train

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
rfc = RandomForestClassifier(verbose=10, n_estimators=10, n_jobs=-1)
rfc.fit(X_train,y_train)

# joblib.dump(rfc, "rfc_model.pkl")

# Use pre-trained model
#rfc = joblib.load("rfc_model.pkl")
#rfc

## Evaluate

In [ ]:
%%time
rfc.verbose = 1 # reduce verbosity
y_hat = rfc.predict(X_test)
print(classification_report(y_test, y_hat))

In [ ]:
%%time
# test single prediction time
pred_times = []
rfc = joblib.load("rfc_model.pkl")
rfc.verbose=0
for i in range(0,100):
    t0=time.time()
    rfc.predict(X_test[i])
    pred_times.append(time.time()-t0)

In [ ]:
np.average(pred_times)

## Model pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

mod_pipeline = Pipeline([
        ('tfidf', tfidf_vectorizer),
        ('rfc', rfc),
    ])

test = [
    "Mike is a terrible data scientist", #NEGATIVE
    "Mike isn't the best data scientist", #POSITIVE
    "Mike is the best data scientist ever", #POSITIVE
    "Mike is a data scientist" #NEUTRAL
       ]

# FULL PIPELINE
test_processed = preprocess_all(test)
preds = mod_pipeline.predict_proba(test_processed)
preds

In [ ]:
PROBA_THRESHOLD = 0.6
def decode_proba(proba):
    if proba[0]  >  PROBA_THRESHOLD:
        return 'NEGATIVE'
    elif proba[1] > PROBA_THRESHOLD:
        return 'POSITIVE'
    else:
        return "NEUTRAL"

def decode_probas(probas):
    return [decode_proba(proba) for proba in probas]

decode_probas(preds)

In [ ]:
joblib.dump(mod_pipeline, 'full_model.pkl')

# Save model in bucket


In [ ]:
import boto3

s3 = boto3.resource('s3')
buckets = list(s3.buckets.all())
buckets[-4:]

In [ ]:
mod_bucket = buckets[-1]
mod_bucket

In [ ]:
filename = 'full_model.pkl'
resp = mod_bucket.upload_file(filename, Key=filename)
resp

## Test model download

In [ ]:
import boto3
BUCKET_NAME="uoft-ml-models"
MODEL_FILE_NAME="full_model.pkl"

def getModel():
    S3 = boto3.client('s3')
    bucket=BUCKET_NAME
    model_file=MODEL_FILE_NAME
    local_file='/tmp/'+model_file
    response = S3.download_file(bucket,model_file,local_file)
    return joblib.load(local_file)


In [ ]:
%%time
mod = getModel()

In [ ]:
import json
json.dumps({"hello":"world"})